# Task 3: Shortest Path with Multiple Agents

This notebook implements a Genetic Algorithm (GA) to find the optimal coordinated paths for multiple agents moving simultaneously through a directed graph, adhering to complex time and collision constraints.

## 1. Design Overview

| Component | Description | Task 3 Implementation |
| :--- | :--- | :--- |
| **Chromosome** | A flat array representing the sequence of actions for all $m$ agents. | `[Agent1_Action1, ..., Agent1_ActionN, Agent2_Action1, ...]` |
| **Gene** | An action: either a **Move** to an adjacent node (Node ID) or an explicit **Wait** (`WAIT_ACTION_CODE`). | Genes $\in \{1, 2, \dots, N\} \cup \{\text{WAIT}\}$ |
| **Fitness Function** | Simulates all agents step-by-step, tracking time and location for **collision detection** and **target sequence validation**. | $\text{Maximize: } - (\text{Total Time} + \text{Collision Penalty} + \text{Order Penalty})$ |
| **Constraints** | Each move takes **$d+10$** time units (distance $d$, plus 10 units of post-move wait at the node). Explicit wait takes **20** units at a node. No two agents can occupy the same **node or edge** simultaneously. | Collision avoidance is enforced via a **large penalty** in the fitness function. |


In [ ]:
import numpy as np
import pygad
from typing import Dict, List, Tuple

# --- 1. CONFIGURATION AND DATA ---

# Example Graph Representation
GRAPH: Dict[int, Dict[int, int]] = {
    1: {2: 5, 3: 2},
    2: {3: 3, 5: 1},
    3: {1: 1, 4: 4},
    4: {5: 2},
    5: {1: 10}
}
NUM_NODES = len(GRAPH)
WAIT_ACTION_CODE = NUM_NODES + 1 # Special code for 'WAIT' action (e.g., 6 if max node is 5).

# Task Parameters
NUM_AGENTS = 2
AGENT_STARTS = [1, 3]           # Starting nodes for Agent 1 and Agent 2
TARGET_SEQUENCE_INITIAL = [2, 5]  # The sequence of nodes that must be visited in order (Immutable)
MAX_STEPS_PER_AGENT = 10        # Maximum actions (moves or waits) an agent can take

# GA Parameters
GENES_PER_AGENT = MAX_STEPS_PER_AGENT
GENE_SPACE = list(range(1, NUM_NODES + 1)) + [WAIT_ACTION_CODE]
SOL_PER_POP = 50
NUM_GENERATIONS = 100
MAX_CHROMOSOME_LENGTH = NUM_AGENTS * GENES_PER_AGENT

print(f"Nodes: {list(GRAPH.keys())}, WAIT Code: {WAIT_ACTION_CODE}")
print(f"Total Chromosome Length: {MAX_CHROMOSOME_LENGTH} ({NUM_AGENTS} agents * {GENES_PER_AGENT} steps)")


## 2. Core Utility Functions: Event Generation

The `generate_events` function simulates an agent's path over time, producing a list of time-stamped location events. This is the **state-time history** used to detect collisions.

### Event Structure
$$(agent\_id, \text{location (Node/Edge)}, \text{start\_time}, \text{end\_time}, \text{location\_type})$$

In [ ]:
def generate_events(agent_id: int, path_segment: np.ndarray, start_node: int) -> Tuple[List[Tuple], int, int]:
    """
    Simulates a single agent's movement according to Task 3 timing rules.
    """
    events = []
    current_time = 0
    current_node = start_node
    
    for action in path_segment:
        
        if action == WAIT_ACTION_CODE:
            # 1. WAIT Action: Cost 20. Occupies current node.
            time_cost = 20
            events.append((agent_id, current_node, current_time, current_time + time_cost, 'NODE'))
            current_time += time_cost
        
        elif action in GRAPH.get(current_node, {}):
            # 2. MOVE Action (Valid)
            next_node = int(action)
            distance = GRAPH[current_node][next_node]
            
            # A. Edge Occupation (distance time units)
            edge_time = distance
            edge_location = tuple(sorted((current_node, next_node))) # Edge (A,B) == (B,A)
            events.append((agent_id, edge_location, current_time, current_time + edge_time, 'EDGE'))
            current_time += edge_time
            
            # B. Node Wait Occupation (10 time units post-move)
            node_wait_time = 10
            events.append((agent_id, next_node, current_time, current_time + node_wait_time, 'NODE'))
            current_time += node_wait_time
            
            current_node = next_node
            
        else:
            # 3. INVALID MOVE: Agent waits at current node. Penalty cost 10.
            time_cost = 10
            events.append((agent_id, current_node, current_time, current_time + time_cost, 'NODE'))
            current_time += time_cost
            
    return events, current_node, current_time


## 3. Fitness Function

The `fitness_func` is responsible for evaluating the quality of the multi-agent plan. It checks three primary criteria, minimizing the total cost:

1. **Total Time:** The time until the last agent completes its assigned path segment.
2. **Collision Penalty:** Calculated by summing the total time units of overlap where two agents share the same node or edge, multiplied by a large factor ($K_c=1000$).
3. **Target Sequence Penalty:** A massive penalty ($K_{\text{invalid}}=1,000,000$) is applied if the `TARGET_SEQUENCE_INITIAL` is not fully completed in order by the collective movements of all agents.


In [ ]:
def fitness_func(ga_instance: pygad.GA, solution: np.ndarray, solution_idx: int) -> float:
    """
    Calculates fitness based on Time, Collisions, and Target Order.
    """
    all_events = []
    target_sequence_tracker = list(TARGET_SEQUENCE_INITIAL)
    max_agent_time = 0
    
    # --- 1. Simulate Agents and Track Target Visits ---
    for i in range(NUM_AGENTS):
        start_index = i * GENES_PER_AGENT
        agent_path_segment = solution[start_index:start_index + GENES_PER_AGENT]
        agent_start_node = AGENT_STARTS[i]
        
        events, _, final_time = generate_events(i, agent_path_segment, agent_start_node)
        all_events.extend(events)
        max_agent_time = max(max_agent_time, final_time)
        
        # Check for Target Sequence Completion
        current_node = agent_start_node
        for action in agent_path_segment:
            if action != WAIT_ACTION_CODE and action in GRAPH.get(current_node, {}):
                current_node = int(action)
                
                if target_sequence_tracker and current_node == target_sequence_tracker[0]:
                    target_sequence_tracker.pop(0) # Target visited

    # --- 2. Collision Check (O(N^2) on number of events) ---
    collision_overlap_time = 0
    COLLISION_PENALTY_FACTOR = 1000 
    
    for i in range(len(all_events)):
        for j in range(i + 1, len(all_events)):
            e1 = all_events[i]
            e2 = all_events[j]
            
            if e1[0] == e2[0]: continue # Skip same agent
            if e1[1] != e2[1]: continue # Skip different location
                
            # Time overlap check: max(start) < min(end)
            overlap_start = max(e1[2], e2[2])
            overlap_end = min(e1[3], e2[3])
                
            if overlap_start < overlap_end:
                collision_overlap_time += (overlap_end - overlap_start)

    # --- 3. Final Cost Calculation ---
    TARGET_PENALTY_FACTOR = 1000000
    invalid_path_penalty = 0
    total_time = 0
    
    if target_sequence_tracker: 
         invalid_path_penalty = TARGET_PENALTY_FACTOR * len(target_sequence_tracker)
         total_time = 10000000 
    else:
        total_time = max_agent_time

    # Total Cost = Time + Collision Penalty + Target Order Penalty
    total_cost = total_time + (collision_overlap_time * COLLISION_PENALTY_FACTOR) + invalid_path_penalty
    
    # Fitness is the negative of the cost (since pygad maximizes fitness)
    fitness = -total_cost
    
    return fitness


## 4. Custom Genetic Operators

Custom operators are necessary to maintain the segmented chromosome structure and efficiently explore the space of cooperative paths, especially focusing on introducing or removing **Wait** actions to resolve predicted collisions.


In [ ]:
def custom_crossover(parents: np.ndarray, offspring_size: Tuple[int], ga_instance: pygad.GA) -> np.ndarray:
    """
    Implements Agent Role Swap (swapping entire paths) and Segment Crossover.
    """
    offspring = []
    
    for k in range(offspring_size[0]):
        parent1 = parents[k % parents.shape[0], :].copy()
        parent2 = parents[(k + 1) % parents.shape[0], :].copy()
        
        # 50% chance for Agent Role Swap
        if np.random.rand() < 0.5 and NUM_AGENTS >= 2:
            agent_to_swap_1 = np.random.randint(NUM_AGENTS)
            agent_to_swap_2 = np.random.randint(NUM_AGENTS)
            
            if agent_to_swap_1 != agent_to_swap_2:
                idx1_start = agent_to_swap_1 * GENES_PER_AGENT
                idx2_start = agent_to_swap_2 * GENES_PER_AGENT
                
                # Swap the entire path segment
                parent1_temp = parent1[idx1_start:idx1_start + GENES_PER_AGENT].copy()
                parent1[idx1_start:idx1_start + GENES_PER_AGENT] = parent2[idx2_start:idx2_start + GENES_PER_AGENT]
                parent2[idx2_start:idx2_start + GENES_PER_AGENT] = parent1_temp
        
        # 50% chance for Segment Crossover
        else:
            agent_idx = np.random.randint(NUM_AGENTS)
            start = agent_idx * GENES_PER_AGENT
            end = start + GENES_PER_AGENT
            
            # Pick a crossover point within that segment
            crossover_point = np.random.randint(start + 1, end)
            
            # Apply single-point crossover within the chosen agent segment
            parent1[crossover_point:end] = parent2[crossover_point:end]
        
        offspring.append(parent1)

    return np.array(offspring)

def custom_mutation(offspring: np.ndarray, ga_instance: pygad.GA) -> np.ndarray:
    """
    Mutates by manipulating the WAIT gene or perturbing a move action.
    """
    for chromosome in offspring:
        
        # 50% chance for Wait Action Mutation (Add/Remove Wait)
        if np.random.rand() < 0.5:
            idx = np.random.randint(0, MAX_CHROMOSOME_LENGTH)
            
            if chromosome[idx] == WAIT_ACTION_CODE:
                # Removal: Change WAIT to a random node (speeding up)
                chromosome[idx] = np.random.choice(list(range(1, NUM_NODES + 1)))
            else:
                # Insertion: Change MOVE to WAIT (collision avoidance)
                chromosome[idx] = WAIT_ACTION_CODE
        
        # 50% chance for Route Perturbation
        else:
            # Select a random gene index corresponding to a move (not WAIT)
            valid_indices = [i for i, x in enumerate(chromosome) if x != WAIT_ACTION_CODE]
            if valid_indices:
                idx = np.random.choice(valid_indices)
                # Change the move to a different, random node index
                chromosome[idx] = np.random.choice(list(range(1, NUM_NODES + 1)))

    return offspring


In [ ]:
def initial_population() -> np.ndarray:
    """Generates a population of random paths respecting the gene space."""
    initial_pop = []
    for _ in range(SOL_PER_POP):
        chromosome = []
        for i in range(NUM_AGENTS):
            path_segment = np.random.choice(GENE_SPACE, size=GENES_PER_AGENT, replace=True)
            chromosome.extend(path_segment)
        initial_pop.append(chromosome)
    return np.array(initial_pop)


## 5. GA Execution

Configure and run the Genetic Algorithm instance using the custom fitness function and operators defined above. You can uncomment `ga_instance.run()` to execute the optimization.

In [ ]:
print("\n--- GA Setup ---")

ga_instance = pygad.GA(
    num_generations=NUM_GENERATIONS,
    num_parents_mating=int(SOL_PER_POP / 2),
    fitness_func=fitness_func,
    initial_population=initial_population(),
    sol_per_pop=SOL_PER_POP,
    num_genes=MAX_CHROMOSOME_LENGTH,
    gene_space=GENE_SPACE,
    
    crossover_type=custom_crossover,
    mutation_type=custom_mutation,
    
    parent_selection_type="rws", 
    crossover_probability=0.8,
    mutation_probability=0.15,
    save_best_solutions=True,
    stop_criteria="saturate_20" 
)

print(f"Starting GA with {NUM_AGENTS} agents, target sequence: {TARGET_SEQUENCE_INITIAL}")

# ga_instance.run() # UNCOMMENT THIS LINE TO START THE OPTIMIZATION

# NOTE: You will need to re-run the cell after uncommenting. 


## 6. Solution Interpretation

This helper function takes the best numerical chromosome found by the GA and converts it into a clear, time-stamped log of actions for your report. (Run this cell after executing the GA in the previous cell).

In [ ]:
def interpret_solution(solution: np.ndarray) -> Dict[str, Dict]:
    """
    Converts the flat chromosome into readable, time-stamped paths for each agent.
    """
    agent_paths_interpreted = {}
    
    for i in range(NUM_AGENTS):
        start_index = i * GENES_PER_AGENT
        agent_segment = solution[start_index:start_index + GENES_PER_AGENT]
        
        current_time = 0
        current_node = AGENT_STARTS[i]
        actions_with_timing = []
        
        for action in agent_segment:
            start_t = current_time
            
            if action == WAIT_ACTION_CODE:
                time_cost = 20
                end_t = current_time + time_cost
                actions_with_timing.append(f"Wait at Node {current_node}: T {start_t} to {end_t}")
                current_time = end_t
            
            elif action in GRAPH.get(current_node, {}):
                next_node = int(action)
                distance = GRAPH[current_node][next_node]
                
                edge_end_t = current_time + distance
                node_end_t = edge_end_t + 10 # Post-move wait
                
                actions_with_timing.append(f"Move {current_node}->{next_node} (Edge T {start_t} to {edge_end_t}) + Wait at {next_node} (Node T {edge_end_t} to {node_end_t})")
                current_time = node_end_t
                current_node = next_node
            
            else:
                # Invalid Move (Forced wait penalty)
                time_cost = 10
                end_t = current_time + time_cost
                actions_with_timing.append(f"Invalid Move: Forced wait at Node {current_node}: T {start_t} to {end_t}")
                current_time = end_t
        
        agent_paths_interpreted[f"Agent {i+1}"] = {
            "Start Node": AGENT_STARTS[i],
            "Final Time": current_time,
            "Actions": actions_with_timing
        }
    
    return agent_paths_interpreted

# Example of post-run analysis (Uncomment and replace with actual results after running GA):
# best_solution_genes, best_fitness, _ = ga_instance.best_solution()
# interpreted_result = interpret_solution(best_solution_genes)

# print("\n--- Interpreted Best Coordinated Path ---")
# for agent, data in interpreted_result.items():
#     print(f"\n🤖 {agent} (Start: {data['Start Node']}, Final Time: {data['Final Time']})")
#     for action in data['Actions']:
#         print(f"  > {action}")
#     print("-" * 20)